# Практическая работа 10. Задание 1
Раскраска графа: жадный алгоритм, Вельш-Пауэлл, хроматическое число

In [1]:
from itertools import product

graph = {
    'A': ['B', 'C', 'D', 'E'],
    'B': ['A', 'C', 'F'],
    'C': ['A', 'B', 'D', 'F', 'G'],
    'D': ['A', 'C', 'E', 'G'],
    'E': ['A', 'D', 'H'],
    'F': ['B', 'C', 'G', 'H'],
    'G': ['C', 'D', 'F', 'H'],
    'H': ['E', 'F', 'G']
}


def greedy_coloring(graph, order=None):
    if order is None:
        order = list(graph.keys())
    colors = {}
    for v in order:
        used = {colors[nb] for nb in graph[v] if nb in colors}
        color = 0
        while color in used:
            color += 1
        colors[v] = color
    return colors


def welsh_powell(graph):
    # сортируем по убыванию степени и собираем букеты
    order = sorted(graph, key=lambda v: -len(graph[v]))
    colors = {}
    color = 0
    for v in order:
        if v in colors:
            continue
        colors[v] = color
        group = [v]
        for u in order:
            if u not in colors and all(u not in graph[x] for x in group):
                colors[u] = color
                group.append(u)
        color += 1
    return colors, order


def is_correct(graph, colors):
    return all(colors[u] != colors[v] for u in graph for v in graph[u])


def bouquets(colors):
    # номер букета: вершины этого цвета
    res = {}
    for c in sorted(set(colors.values())):
        res[c + 1] = [v for v in sorted(colors) if colors[v] == c]
    return res


def can_color(graph, k):
    # перебор всех раскрасок в k цветов
    vs = list(graph)
    for c in product(range(k), repeat=len(vs)):
        colors = dict(zip(vs, c))
        if is_correct(graph, colors):
            return colors
    return None

In [2]:
col = greedy_coloring(graph)
print('Жадный (порядок A..H):', col)
print('  цветов:', max(col.values()) + 1)
print('  раскраска правильная:', is_correct(graph, col))

wp, order = welsh_powell(graph)
print('Вельш-Пауэлл, порядок по степени:', order)
print('  букеты:', bouquets(wp))
print('  цветов:', max(wp.values()) + 1)
print('  раскраска правильная:', is_correct(graph, wp))

Жадный (порядок A..H): {'A': 0, 'B': 1, 'C': 2, 'D': 1, 'E': 2, 'F': 0, 'G': 3, 'H': 1}
  цветов: 4
  раскраска правильная: True
Вельш-Пауэлл, порядок по степени: ['C', 'A', 'D', 'F', 'G', 'B', 'E', 'H']
  букеты: {1: ['C', 'E'], 2: ['A', 'F'], 3: ['B', 'D', 'H'], 4: ['G']}
  цветов: 4
  раскраска правильная: True


In [3]:
from itertools import combinations

# нижняя оценка: самая большая клика (перебор подмножеств)
big = []
for k in range(1, len(graph) + 1):
    for c in combinations(graph, k):
        if all(v in graph[u] for u, v in combinations(c, 2)):
            big = list(c)
            break
print('Наибольшая клика:', big, ', значит chi(G) >=', len(big))

# можно ли меньше цветов: перебираем k = 1, 2, 3, ...
for k in range(1, 5):
    res = can_color(graph, k)
    print(k, 'цвет(а):', 'можно' if res else 'нельзя')
    if res:
        print('Хроматическое число =', k)
        print('Пример раскраски:', bouquets(res))
        break

Наибольшая клика: ['A', 'B', 'C'] , значит chi(G) >= 3
1 цвет(а): нельзя
2 цвет(а): нельзя
3 цвет(а): нельзя
4 цвет(а): можно
Хроматическое число = 4
Пример раскраски: {1: ['A', 'F'], 2: ['B', 'D', 'H'], 3: ['C', 'E'], 4: ['G']}
